# MAG7 five-shot headline relevance: manual sample audit

This notebook runs the existing five-shot relevance prompt on a small, balanced sample: up to 40 headlines per MAG7 ticker (280 maximum). It is for manual inspection before any full-dataset run. It does not use tone, event extraction, or price data.


In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 1000)

DATASET_REPO = "cookekieran/alphavantage-market-news"
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]
MONTHS = pd.period_range("2023-01", "2026-06", freq="M").astype(str).tolist()
SAMPLE_PER_TICKER = 40
AUDIT_PER_LABEL = 15
RANDOM_SEED = 42
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for candidate in (PROJECT_ROOT / ".env", PROJECT_ROOT / "env.txt"):
    if candidate.exists():
        load_dotenv(candidate)

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is required."
assert torch.cuda.is_available(), "CUDA GPU required. Do not run this notebook on CPU."

print({"sample_per_ticker": SAMPLE_PER_TICKER, "maximum_headlines": len(TICKERS) * SAMPLE_PER_TICKER, "model": MODEL_ID})


In [ ]:
article_frames = []

for month in MONTHS:
    path = Path(
        hf_hub_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            filename=f"articles/{month}.parquet",
            token=HF_TOKEN,
        )
    )
    article_frames.append(pd.read_parquet(path))

articles = pd.concat(article_frames, ignore_index=True)

required = {"article_uid", "requested_entity", "time_published", "title"}
missing = required - set(articles.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

all_mag7_articles = (
    articles.loc[
        articles["requested_entity"].isin(TICKERS),
        ["article_uid", "requested_entity", "time_published", "title"],
    ]
    .dropna(subset=["time_published", "title"])
    .drop_duplicates(["requested_entity", "article_uid"])
    .assign(time_published=lambda frame: pd.to_datetime(frame["time_published"], utc=True))
    .sort_values(["requested_entity", "time_published", "article_uid"])
)

sampled_articles = (
    all_mag7_articles
    .groupby("requested_entity", group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(SAMPLE_PER_TICKER, len(group)),
            random_state=RANDOM_SEED,
        )
    )
    .sort_values(["requested_entity", "time_published", "article_uid"])
    .reset_index(drop=True)
)

display(
    sampled_articles.groupby("requested_entity", as_index=False)
    .size()
    .rename(columns={"requested_entity": "ticker", "size": "sampled_headlines"})
)
print(f"Headlines to classify: {len(sampled_articles):,}")


In [ ]:
FEW_SHOT_EXAMPLES = [
    ("2023-01-03", "Apple reports a sharp iPhone sales decline and cuts its revenue outlook", "relevant"),
    ("2023-01-04", "Technology ETF increases holdings in Apple, Microsoft and Nvidia", "not relevant"),
    ("2023-01-05", "Apple supplier reports disruption to iPhone component production", "relevant"),
    ("2023-01-06", "Five mega-cap technology stocks investors are watching this week", "not relevant"),
    ("2023-01-09", "Apple TV+ renews a drama series for another season", "not relevant"),
]

def build_relevance_prompt(ticker, published_at, title):
    demonstrations = "\n\n".join(
        f"Publication date: {date}\nHeadline: {headline}\nClassification: {label}"
        for date, headline, label in FEW_SHOT_EXAMPLES
    )

    return f"""You classify financial-news headlines for ticker {ticker}.

Classify a headline as relevant only when it describes a specific, company-level development
that could plausibly affect {ticker}'s revenue, demand, costs, supply chain, products, strategy,
financing, competitive position, or regulatory risk.

Classify it as not relevant when {ticker} is only in an ETF, index, stock list, market roundup,
peer comparison, analyst recommendation, generic sector commentary, review, minor corporate
update, or indirect supplier/competitor story without a direct company-specific mechanism.

Do not predict a return or use information outside the date and headline.
Reply with exactly one label: relevant or not relevant.

LABELLED EXAMPLES
{demonstrations}

TARGET HEADLINE
Publication date: {published_at.date().isoformat()}
Headline: {title}
Classification:""".strip()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype="auto",
    device_map="auto",
).eval()

def classify_relevance(prompt):
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    encoded = tokenizer(chat, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw_response = tokenizer.decode(
        generated[0, encoded["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    labels = re.findall(r"\bnot relevant\b|\brelevant\b", raw_response.lower())
    return (labels[-1] if labels else None), raw_response


In [ ]:
records = []

for ticker, ticker_articles in sampled_articles.groupby("requested_entity"):
    print(f"Classifying {ticker}: {len(ticker_articles)} headlines")

    for article in ticker_articles.itertuples(index=False):
        classification, raw_response = classify_relevance(
            build_relevance_prompt(
                ticker,
                article.time_published,
                article.title,
            )
        )

        records.append({
            "ticker": ticker,
            "article_uid": article.article_uid,
            "time_published": article.time_published,
            "title": article.title,
            "classification": classification,
            "raw_response": raw_response,
        })

sample_results = pd.DataFrame(records)

display(
    sample_results.groupby(["ticker", "classification"], dropna=False, as_index=False)
    .size()
    .rename(columns={"size": "headlines"})
)


In [ ]:
# Display up to 30 audit examples per ticker: 15 relevant and 15 not relevant.
audit_parts = []

for ticker in TICKERS:
    ticker_results = sample_results.loc[sample_results["ticker"].eq(ticker)]

    for label in ["relevant", "not relevant"]:
        candidates = ticker_results.loc[
            ticker_results["classification"].eq(label)
        ]

        audit_parts.append(
            candidates.sample(
                n=min(AUDIT_PER_LABEL, len(candidates)),
                random_state=RANDOM_SEED,
            )
        )

manual_audit = (
    pd.concat(audit_parts, ignore_index=True)
    .sort_values(["ticker", "classification", "time_published"])
)

for ticker in TICKERS:
    print(f"\n{'=' * 15} {ticker}: manual relevance audit {'=' * 15}")
    display(
        manual_audit.loc[
            manual_audit["ticker"].eq(ticker),
            ["time_published", "classification", "title"],
        ]
    )

print("Check both labels: relevant rows reveal false positives; not-relevant rows reveal false negatives.")
